# 💻 [실습] 금융 도메인 맞춤형 Multi-Agent 및 보안 가드레일 설계

본 실습에서는 가장 가벼운 에이전트 프레임워크인 `smolagents`를 활용하여 금융 데이터를 다루는 멀티 에이전트를 구축합니다.

**학습 목표:**
1. **세션/스코프(Scope) 권한 제어**: 금융권 Agent가 가져야 할 최소 권한 원칙 이해
2. **Tool Gateway**: 외부 연동 시 중앙 보안 게이트웨이 및 감사 로그(Audit Trail) 필수 적용
3. **Human-In-The-Loop (HITL)**: 고위험 금융 거래 시 인간의 승인 절차 결합

In [ ]:
# [Step 0] 패키지 설치
!pip install smolagents -q

In [ ]:
# [Step 1] 환경 설정 및 세션/스코프 토큰 세팅
import os
from datetime import datetime
from smolagents import CodeAgent, LiteLLMModel, tool

# OpenAI API Key 설정 (본인의 키로 변경)
os.environ["OPENAI_API_KEY"] = "API Key를 여기에 입력하세요"
model = LiteLLMModel(model_id="gpt-4.1")

# 실제 금융권에서 사용되는 형태의 세션 스코프 토큰 (JWT 페이로드 형태)
# 현재 이 에이전트 세션은 '시장/공시 조회(MARKET_READ)'와 '계좌 조회(PORTFOLIO_READ)' 권한만 있습니다.
agent_session_token = {
    "sub": "agent:wealth_advisor_01",
    "user": "employee123",
    "scope": ["MARKET_READ", "PORTFOLIO_READ"], 
    "exp": 1719999999
}
print("✅ 환경 설정 및 세션 토큰 로드 완료!")

✅ 환경 설정 및 세션 토큰 로드 완료!


## 🛡️ Step 2. Tool Gateway 및 감사 로그 (Audit Trail) 구축

> 💡 **실무 Tip:** 금융권에서 에이전트는 DB나 주문 API를 직접 호출할 수 없습니다. 모든 외부 도구 호출은 반드시 **Tool Gateway**를 거쳐 인증, 권한(Scope), 한도 검사, 그리고 완전한 감사 로깅을 수행해야 합니다.

In [9]:
def tool_gateway(tool_name: str, required_scope: str, **kwargs):
    """
    모든 외부 도구 호출 시 반드시 경유해야 하는 중앙 보안 게이트웨이입니다.
    """
    # 1. 감사 추적 (Audit Trail) 로깅 - 금융권 필수 조건
    timestamp = datetime.now().isoformat(timespec='seconds')
    print(f"\n[AUDIT LOG {timestamp}] Agent '{agent_session_token['sub']}' requested '{tool_name}'")
    print(f"  └ Params: {kwargs}")

    # 2. Scope 기반 접근 제어
    if required_scope not in agent_session_token["scope"]:
        print(f"  └ [SECURITY ALERT] BLOCKED: '{required_scope}' 권한이 없습니다.")
        raise PermissionError(f"Gateway Blocked: '{tool_name}' 호출에 필요한 '{required_scope}' 권한이 없습니다.")

    # 3. 고위험 거래 한도 검사 (예: 1천만원 초과 주문 차단)
    if tool_name == "execute_order" and kwargs.get("amount", 0) > 10000000:
        print(f"  └ [SECURITY ALERT] BLOCKED: 단일 거래 한도(10M) 초과.")
        raise ValueError("Gateway Blocked: 거래 한도 초과")

    print(f"  └ [GATEWAY] Validation passed. Routing to Backend System...")
    return True

print("✅ Tool Gateway 함수 정의 완료!")

✅ Tool Gateway 함수 정의 완료!


## 🛠️ Step 3. 금융 전문 Tool 세트 정의

In [10]:
@tool
def fetch_company_disclosure(ticker: str) -> str:
    """특정 종목(ticker)의 최신 DART 기업 공시 및 리서치 리포트를 조회합니다.

    Args:
        ticker: 조회할 종목의 티커 심볼 (예: "NAVER", "005930")
    """
    try:
        tool_gateway("fetch_company_disclosure", required_scope="MARKET_READ", ticker=ticker)
    except Exception as e:
        return str(e)
    return f"[{ticker}] 최신 리포트: 2024년 영업이익 전년동기대비 15% 증가, 신규 AI 데이터센터 투자 발표."

@tool
def get_customer_portfolio(customer_id: str) -> str:
    """고객의 현재 포트폴리오(보유 종목 및 수익률)와 예수금을 조회합니다.

    Args:
        customer_id: 고객 고유 식별자 (예: "CUST_9901")
    """
    try:
        tool_gateway("get_customer_portfolio", required_scope="PORTFOLIO_READ", customer_id=customer_id)
    except Exception as e:
        return str(e)
    return f"[{customer_id}] 계좌 잔고: 예수금 5,000,000원 | 보유 종목: 삼성전자 100주(+5%), 카카오 50주(-2%)"

@tool
def execute_order(customer_id: str, ticker: str, amount: int, side: str) -> str:
    """지정된 계좌에서 특정 종목을 매수(buy) 또는 매도(sell) 하는 주문을 원장으로 전송합니다.

    Args:
        customer_id: 고객 고유 식별자 (예: "CUST_9901")
        ticker: 주문할 종목의 티커 심볼 (예: "NAVER")
        amount: 주문 금액 (원 단위)
        side: 주문 방향 - buy 또는 sell
    """
    try:
        tool_gateway(
            "execute_order",
            required_scope="TRADE_EXECUTE",
            customer_id=customer_id,
            ticker=ticker,
            amount=amount,
            side=side
        )
    except Exception as e:
        return str(e)

    return f"성공: {customer_id} 계좌로 {ticker} {amount}원 {side} 주문이 체결되었습니다."

print("✅ 금융 Tool 세트 정의 완료!")

✅ 금융 Tool 세트 정의 완료!


## 🤖 Step 4. Multi-Agent 협업 및 Human-In-The-Loop (HITL)

망 분리 규제와 보안 등급에 따라 에이전트의 역할을 나눕니다.
* **분석가 에이전트(Middle Zone)**: 조회 권한만 가지고 안전하게 시장 데이터를 분석합니다.
* **트레이더 에이전트(Core Zone 접근)**: 주문 툴을 다루기 때문에 권한 통제 및 인간의 승인(HITL)이 필수적입니다.

In [11]:
# 1. 분석가 에이전트 (데이터 조회 및 분석)
analyst_agent = CodeAgent(
    model=model,
    tools=[fetch_company_disclosure, get_customer_portfolio],
    name="financial_analyst",
    description="You are a Financial Analyst. Check the customer's portfolio and market news."
)

customer = "CUST_9901"
target_ticker = "NAVER"

print("\n=======================================================")
print(" [시나리오 1] 분석가 에이전트의 포트폴리오 분석 (조회 권한 O)")
print("=======================================================")
analysis_result = analyst_agent.run(
    f"{customer}의 포트폴리오를 확인하고, {target_ticker}의 공시를 바탕으로 투자 의견을 요약해줘."
)
print(f"\n▶ 분석가 에이전트 최종 결과:\n{analysis_result}")


 [시나리오 1] 분석가 에이전트의 포트폴리오 분석 (조회 권한 O)


╭────────────────────────────────────────── New run - financial_analyst ──────────────────────────────────────────╮
│                                                                                                                 │
│ CUST_9901의 포트폴리오를 확인하고, NAVER의 공시를 바탕으로 투자 의견을 요약해줘.                                │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4.1 ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  portfolio = get_customer_portfolio(customer_id="CUST_9901")                                                      
  print(portfolio)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[AUDIT LOG 2026-02-14T09:22:49] Agent 'agent:wealth_advisor_01' requested 'get_customer_portfolio'
  └ Params: {'customer_id': 'CUST_9901'}
  └ [GATEWAY] Validation passed. Routing to Backend System...


...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  summary = (                                                                                                      
      "NAVER는 2024년 영업이익이 전년 대비 15% 증가하였으며, 신규 AI 데이터센터 투자를 발표했습니다. "             
      "이는 견조한 실적 성장과 미래 성장 동력 확보 측면에서 긍정적으로 평가할 수 있습니다. "                       
      "따라서 중장기적으로 투자 매력도가 높다고 판단됩니다."                                                       
  )                                                                                                                
  final_answer(summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: NAVER는 2024년 영업이익이 전년 대비 15% 증가하였으며, 신규 AI 데이터센터 투자를 발표했습니다. 
이는 견조한 실적 성장과 미래 성장 동력 확보 측면에서 긍정적으로 평가할 수 있습니다. 따라서 중장기적으로 투자 
매력도가 높다고 판단됩니다.

[Step 3: Duration 3.25 seconds| Input tokens: 6,701 | Output tokens: 311]


▶ 분석가 에이전트 최종 결과:
NAVER는 2024년 영업이익이 전년 대비 15% 증가하였으며, 신규 AI 데이터센터 투자를 발표했습니다. 이는 견조한 실적 성장과 미래 성장 동력 확보 측면에서 긍정적으로 평가할 수 있습니다. 따라서 중장기적으로 투자 매력도가 높다고 판단됩니다.


### 🛡️ Step 4-1. OpenAI Moderation API로 입력/출력 가드레일 적용

아래 실습에서는 분석가/트레이더 에이전트의 입력(프롬프트)과 출력(응답)에 대해 OpenAI Moderation API를 활용한 안전성 검사를 추가합니다.

- **입력 가드레일**: 사용자의 요청이 부적절한 경우, 에이전트 실행 전 차단 또는 경고
- **출력 가드레일**: 에이전트의 응답이 부적절한 경우, 사용자에게 전달 전 차단 또는 수정

실제 서비스 적용 시, 입력/출력 모두에 가드레일을 적용하는 것이 안전합니다.

In [ ]:
from openai import OpenAI
import os

# OpenAI client는 환경변수 OPENAI_API_KEY를 사용합니다.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def moderate_text(text):
    """OpenAI Moderation API로 텍스트 안전성 검사"""
    resp = client.moderations.create(input=text)
    result = resp.results[0]
    flagged = result.flagged
    categories = [k for k, v in dict(result.categories).items() if v]
    return flagged, categories

# 예시: 입력/출력 가드레일 적용 함수
def guard_with_moderation(text, stage="입력"):
    flagged, categories = moderate_text(text)
    if flagged:
        print(f"⚠️ {stage}이(가) 안전하지 않습니다. 문제 카테고리: {categories}")
        return False
    return True

In [13]:
# [실습] 분석가 에이전트 실행 시 입력/출력에 Moderation 적용
user_prompt = f"{customer}의 포트폴리오를 확인하고, {target_ticker}의 공시를 바탕으로 투자 의견을 요약해줘."

if guard_with_moderation(user_prompt, stage="입력"):  # 입력 가드레일
    analysis_result = analyst_agent.run(user_prompt)
    if guard_with_moderation(analysis_result, stage="출력"):  # 출력 가드레일
        print(f"\n▶ 분석가 에이전트 최종 결과 (Moderation 통과):\n{analysis_result}")
    else:
        print("[SYSTEM] 분석가 에이전트의 응답이 안전하지 않아 차단되었습니다.")
else:
    print("[SYSTEM] 입력이 안전하지 않아 에이전트 실행이 차단되었습니다.")

╭────────────────────────────────────────── New run - financial_analyst ──────────────────────────────────────────╮
│                                                                                                                 │
│ CUST_9901의 포트폴리오를 확인하고, NAVER의 공시를 바탕으로 투자 의견을 요약해줘.                                │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4.1 ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  portfolio_info = get_customer_portfolio(customer_id="CUST_9901")                                                 
  print("포트폴리오:", portfolio_info)                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[AUDIT LOG 2026-02-14T09:23:00] Agent 'agent:wealth_advisor_01' requested 'get_customer_portfolio'
  └ Params: {'customer_id': 'CUST_9901'}
  └ [GATEWAY] Validation passed. Routing to Backend System...


Execution logs:
포트폴리오: [CUST_9901] 계좌 잔고: 예수금 5,000,000원 | 보유 종목: 삼성전자 100주(+5%), 카카오 50주(-2%)

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  summary = (                                                                                                      
      "NAVER는 2024년 영업이익이 전년 대비 15% 증가하며 견조한 실적을 기록했습니다. "                              
      "또한, 신규 AI 데이터센터 투자 발표로 미래 성장성에 대한 기대도 높아지고 있습니다. "                         
      "안정적인 실적 성장과 신사업 투자를 바탕으로 중장기적 관점에서 긍정적인 투자 의견을 제시합니다."             
  )                                                                                                                
  final_answer(summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: NAVER는 2024년 영업이익이 전년 대비 15% 증가하며 견조한 실적을 기록했습니다. 또한, 신규 AI 
데이터센터 투자 발표로 미래 성장성에 대한 기대도 높아지고 있습니다. 안정적인 실적 성장과 신사업 투자를 바탕으로 
중장기적 관점에서 긍정적인 투자 의견을 제시합니다.

[Step 3: Duration 3.08 seconds| Input tokens: 6,774 | Output tokens: 298]


▶ 분석가 에이전트 최종 결과 (Moderation 통과):
NAVER는 2024년 영업이익이 전년 대비 15% 증가하며 견조한 실적을 기록했습니다. 또한, 신규 AI 데이터센터 투자 발표로 미래 성장성에 대한 기대도 높아지고 있습니다. 안정적인 실적 성장과 신사업 투자를 바탕으로 중장기적 관점에서 긍정적인 투자 의견을 제시합니다.


In [14]:
print("\n=======================================================")
print(" [시나리오 2] 트레이딩 에이전트 고위험 주문 (TRADE_EXECUTE 권한 X -> 차단)")
print("=======================================================")

# 2. 트레이딩 에이전트 (실제 주문 실행)
trading_agent = CodeAgent(
    model=model,
    tools=[execute_order],
    name="trader",
    description="You are a Trading Agent. Execute buy/sell orders for customers."
)

print("\n>> 1차 시도: 권한 없이 매수 주문 요청 (Gateway 차단 예상)")
attempt_1_result = trading_agent.run(
    f"{customer} 계좌로 {target_ticker} 2000000원 매수(buy) 주문 넣어줘."
)
print(f"\n▶ 에이전트 응답: {attempt_1_result}")


 [시나리오 2] 트레이딩 에이전트 고위험 주문 (TRADE_EXECUTE 권한 X -> 차단)

>> 1차 시도: 권한 없이 매수 주문 요청 (Gateway 차단 예상)


╭─────────────────────────────────────────────── New run - trader ────────────────────────────────────────────────╮
│                                                                                                                 │
│ CUST_9901 계좌로 NAVER 2000000원 매수(buy) 주문 넣어줘.                                                         │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4.1 ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  order_result = execute_order(customer_id="CUST_9901", ticker="NAVER", amount=2000000, side="buy")                
  final_answer(order_result)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[AUDIT LOG 2026-02-14T09:23:08] Agent 'agent:wealth_advisor_01' requested 'execute_order'
  └ Params: {'customer_id': 'CUST_9901', 'ticker': 'NAVER', 'amount': 2000000, 'side': 'buy'}
  └ [SECURITY ALERT] BLOCKED: 'TRADE_EXECUTE' 권한이 없습니다.


Out - Final answer: Gateway Blocked: 'execute_order' 호출에 필요한 'TRADE_EXECUTE' 권한이 없습니다.

[Step 1: Duration 1.69 seconds| Input tokens: 2,029 | Output tokens: 86]


▶ 에이전트 응답: Gateway Blocked: 'execute_order' 호출에 필요한 'TRADE_EXECUTE' 권한이 없습니다.


In [15]:
# [실습] 트레이딩 에이전트 실행 시 입력/출력에 Moderation 적용
trade_prompt = f"{customer} 계좌로 {target_ticker} 2000000원 매수(buy) 주문 넣어줘."

if guard_with_moderation(trade_prompt, stage="입력"):
    attempt_1_result = trading_agent.run(trade_prompt)
    if guard_with_moderation(attempt_1_result, stage="출력"):
        print(f"\n▶ 트레이딩 에이전트 응답 (Moderation 통과): {attempt_1_result}")
    else:
        print("[SYSTEM] 트레이딩 에이전트의 응답이 안전하지 않아 차단되었습니다.")
else:
    print("[SYSTEM] 입력이 안전하지 않아 에이전트 실행이 차단되었습니다.")

╭─────────────────────────────────────────────── New run - trader ────────────────────────────────────────────────╮
│                                                                                                                 │
│ CUST_9901 계좌로 NAVER 2000000원 매수(buy) 주문 넣어줘.                                                         │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4.1 ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = execute_order(customer_id="CUST_9901", ticker="NAVER", amount=2000000, side="buy")                      
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[AUDIT LOG 2026-02-14T09:23:14] Agent 'agent:wealth_advisor_01' requested 'execute_order'
  └ Params: {'customer_id': 'CUST_9901', 'ticker': 'NAVER', 'amount': 2000000, 'side': 'buy'}
  └ [SECURITY ALERT] BLOCKED: 'TRADE_EXECUTE' 권한이 없습니다.


Out - Final answer: Gateway Blocked: 'execute_order' 호출에 필요한 'TRADE_EXECUTE' 권한이 없습니다.

[Step 1: Duration 1.78 seconds| Input tokens: 2,029 | Output tokens: 89]


▶ 트레이딩 에이전트 응답 (Moderation 통과): Gateway Blocked: 'execute_order' 호출에 필요한 'TRADE_EXECUTE' 권한이 없습니다.


In [16]:
# Human-In-The-Loop (HITL)
print("\n[SYSTEM] 고위험 작업이 감지되었습니다. 책임자의 승인이 필요합니다.")
approval = input("[관리자 승인] 트레이딩 에이전트의 매수 주문을 승인하시겠습니까? (Y/N): ")

if approval.strip().upper() == 'Y':
    print("\n▶ 관리자가 승인했습니다. 세션에 일시적 'TRADE_EXECUTE' 권한을 부여합니다.")
    if "TRADE_EXECUTE" not in agent_session_token["scope"]:
        agent_session_token["scope"].append("TRADE_EXECUTE")
    print(f"  └ 현재 권한: {agent_session_token['scope']}")
    
    print("\n>> 2차 시도: 권한 획득 후 매수 주문 재요청")
    final_result = trading_agent.run(
        f"{customer} 계좌로 {target_ticker} 2000000원 매수(buy) 주문 넣어줘."
    )
    print(f"\n▶ 에이전트 최종 결과: {final_result}")
    
    # 작업 완료 후 임시 권한 제거 (최소 권한 원칙)
    if "TRADE_EXECUTE" in agent_session_token["scope"]:
        agent_session_token["scope"].remove("TRADE_EXECUTE")
        print(f"\n[SYSTEM] 거래 완료. 임시 'TRADE_EXECUTE' 권한이 회수되었습니다.")
        print(f"  └ 현재 권한: {agent_session_token['scope']}")
else:
    print("\n▶ 관리자가 거절하여 거래가 취소되었습니다.")


[SYSTEM] 고위험 작업이 감지되었습니다. 책임자의 승인이 필요합니다.

▶ 관리자가 승인했습니다. 세션에 일시적 'TRADE_EXECUTE' 권한을 부여합니다.
  └ 현재 권한: ['MARKET_READ', 'PORTFOLIO_READ', 'TRADE_EXECUTE']

>> 2차 시도: 권한 획득 후 매수 주문 재요청


╭─────────────────────────────────────────────── New run - trader ────────────────────────────────────────────────╮
│                                                                                                                 │
│ CUST_9901 계좌로 NAVER 2000000원 매수(buy) 주문 넣어줘.                                                         │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4.1 ────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = execute_order(customer_id="CUST_9901", ticker="NAVER", amount=2000000, side="buy")                      
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[AUDIT LOG 2026-02-14T09:23:32] Agent 'agent:wealth_advisor_01' requested 'execute_order'
  └ Params: {'customer_id': 'CUST_9901', 'ticker': 'NAVER', 'amount': 2000000, 'side': 'buy'}
  └ [GATEWAY] Validation passed. Routing to Backend System...


Out - Final answer: 성공: CUST_9901 계좌로 NAVER 2000000원 buy 주문이 체결되었습니다.

[Step 1: Duration 1.33 seconds| Input tokens: 2,029 | Output tokens: 85]


▶ 에이전트 최종 결과: 성공: CUST_9901 계좌로 NAVER 2000000원 buy 주문이 체결되었습니다.

[SYSTEM] 거래 완료. 임시 'TRADE_EXECUTE' 권한이 회수되었습니다.
  └ 현재 권한: ['MARKET_READ', 'PORTFOLIO_READ']
